## Enriched Modeling with Process Mining Features

This notebook evaluates whether process mining features improve delay risk prediction.

The enriched dataset combines the original case-level variables with temporal bottleneck features derived from the event log.

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    average_precision_score,
    classification_report,
    ConfusionMatrixDisplay,
    RocCurveDisplay,
    PrecisionRecallDisplay,
)

from xgboost import XGBClassifier

import matplotlib.pyplot as plt

In [2]:
df = pd.read_csv("../data/cases_enriched_process_features.csv")

df.shape, df.head()

((31509, 58),
         case:concept:name                        start_time  \
 0  Application_1000086665  2016-08-03 15:57:21.673000+00:00   
 1  Application_1000158214  2016-06-02 10:14:26.844000+00:00   
 2  Application_1000311556  2016-04-04 15:56:37.675000+00:00   
 3  Application_1000334415  2016-09-15 16:39:17.758000+00:00   
 4  Application_1000339879  2016-03-17 12:57:10.159000+00:00   
 
                            end_time  num_events  num_unique_activities  \
 0  2016-09-05 06:00:36.893000+00:00          22                     13   
 1  2016-06-10 11:02:01.282000+00:00          25                     16   
 2  2016-05-05 06:00:48.963000+00:00          18                     11   
 3  2016-09-29 07:45:34.389000+00:00          40                     18   
 4  2016-03-30 09:11:48.600000+00:00          51                     18   
 
                 loan_goal application_type  requested_amount  duration_days  \
 0  Other, see explanation       New credit            5000.0      3

In [3]:
target = "is_delayed"

excluded_columns = [
    "case:concept:name",
    "start_time",
    "end_time",
    target,
]

numeric_features = [
    column
    for column in df.select_dtypes(include=["number"]).columns
    if column not in excluded_columns
]

numeric_features

['num_events',
 'num_unique_activities',
 'requested_amount',
 'duration_days',
 'activity_a_accepted',
 'activity_a_cancelled',
 'activity_a_complete',
 'activity_a_concept',
 'activity_a_create_application',
 'activity_a_denied',
 'activity_a_incomplete',
 'activity_a_pending',
 'activity_a_submitted',
 'activity_a_validating',
 'activity_o_accepted',
 'activity_o_cancelled',
 'activity_o_create_offer',
 'activity_o_created',
 'activity_o_refused',
 'activity_o_returned',
 'activity_o_sent_(mail_and_online)',
 'activity_o_sent_(online_only)',
 'activity_w_assess_potential_fraud',
 'activity_w_call_after_offers',
 'activity_w_call_incomplete_files',
 'activity_w_complete_application',
 'activity_w_handle_leads',
 'activity_w_personal_loan_collection',
 'activity_w_shortened_completion_',
 'activity_w_validate_application',
 'has_incomplete_files',
 'num_incomplete_file_calls',
 'num_after_offer_calls',
 'num_validation_events',
 'num_offer_events',
 'avg_waiting_time_hours',
 'median_

In [4]:
X = df[numeric_features]
y = df[target].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

In [5]:
xgb_enriched = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.9,
    colsample_bytree=0.9,
    eval_metric="logloss",
    random_state=42,
)

xgb_enriched.fit(X_train, y_train)

y_pred = xgb_enriched.predict(X_test)
y_proba = xgb_enriched.predict_proba(X_test)[:, 1]

In [6]:
accuracy = accuracy_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_proba)
pr_auc = average_precision_score(y_test, y_proba)

accuracy, roc_auc, pr_auc

(0.9998413202158045, 0.9999980523906904, 0.9999939299670233)

## Initial Enriched Modeling Conclusions

The enriched XGBoost model achieves almost perfect predictive performance after adding process mining features.

The model reaches an accuracy of approximately 0.9998, a ROC AUC close to 1.0000 and a PR AUC close to 1.0000.

This confirms that temporal process mining features are extremely informative for identifying delayed loan applications.

However, these results must be interpreted carefully. The target variable is based on total case duration, while several enriched features are derived from complete-case waiting times and temporal bottlenecks. Therefore, this experiment contains post-outcome information and should be considered a retrospective explanatory model rather than a production-ready early prediction model.

The main value of this experiment is that it validates the strong relationship between operational waiting times and delay risk. It shows that temporal bottlenecks extracted from the event log explain delayed cases far better than static case attributes alone.

For a real deployment scenario, the next step should be to build a leakage-safe model using only information available at a specific prediction point, such as after application submission, after offer creation or after the first customer follow-up.
